In [ ]:
import pandas as pd

# Read the CSV file
df_MC = pd.read_csv('/groups/icecube/holgerkc/Thesis_Analysis/old/MC_prediction_muon_noise_neutrino_02_02_2026.csv')
df_BS = pd.read_csv('/groups/icecube/holgerkc/Thesis_Analysis/old/Burnsample_prediction_muon_noise_neutrino_2022.csv')
# Display column names (variables)
print("MC variables (columns):", list(df_MC.columns))
print("BS variables (columns):", list(df_BS.columns))

# Display the DataFrame
print("MC Data:")
print(df_MC)
print("BS Data:")
print(df_BS)

In [ ]:
import sqlite3

def get_event_data(event_no, db_path='/groups/icecube/holgerkc/Thesis_Analysis/old/MC_pulsemap_muon_noise_neutrino_02_02_2026.db'):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    # List all tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall()]
    print("Tables in database:", tables)
    # For each table, try to get rows with the given event_no
    for table in tables:
        try:
            df_table = pd.read_sql_query(f"SELECT * FROM {table} WHERE event_no = ?", conn, params=(event_no,))
            if not df_table.empty:
                print(f"\nData from table '{table}' for event_no {event_no}:")
                print(df_table)
        except Exception as e:
            print(f"Could not query table {table}: {e}")
    conn.close()

# Example usage:
get_event_data(1000000)

In [ ]:
def get_bs_event_data(event_no):
    # Set pandas display options to show all data
    
    db_path = 'file:/lustre/hpc/project/icecube/Burnsample/databases/IC86.22/burnsample_IC8622_merged.db?mode=ro&immutable=1'
    conn = sqlite3.connect(db_path, uri=True, timeout=10)
    cursor = conn.cursor()
    
    # List all tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall()]
    print("Tables in database:", tables)
    
    # Initialize arrays for the columns we want to extract
    charge_array = []
    dom_x_array = []
    dom_y_array = []
    dom_z_array = []
    
    # For each table, try to get rows with the given event_no
    for table in tables:
        try:
            df_table = pd.read_sql_query(f"SELECT * FROM {table} WHERE event_no = ?", conn, params=(event_no,))
            if not df_table.empty:
                print(f"\nData from table '{table}' for event_no {event_no}:")
                print(df_table)
                
                # Extract columns into arrays
                charge_array.extend(df_table['charge'].tolist())
                dom_x_array.extend(df_table['dom_x'].tolist())
                dom_y_array.extend(df_table['dom_y'].tolist())
                dom_z_array.extend(df_table['dom_z'].tolist())
        except Exception as e:
            print(f"Could not query table {table}: {e}")
    
    conn.close()
    
    # Print the extracted arrays
    print("\n=== Extracted Arrays ===")
    print(f"charge: {charge_array}")
    print(f"dom_x: {dom_x_array}")
    print(f"dom_y: {dom_y_array}")
    print(f"dom_z: {dom_z_array}")
    
    return charge_array, dom_x_array, dom_y_array, dom_z_array

# Example usage:
charge, dom_x, dom_y, dom_z = get_bs_event_data(7) #4 #7 

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Scale charge for marker size
sizes = np.array(charge) * 5

# Create 3D scatter plot with Plotly
fig = go.Figure(data=[go.Scatter3d(
    x=dom_x,
    y=dom_y,
    z=dom_z,
    mode='markers',
    marker=dict(
        size=sizes,
        color=charge,
        colorscale='Viridis',
        opacity=0.7,
        colorbar=dict(title="Charge")
    ),
    text=[f"Charge: {c:.3f}" for c in charge],
    hoverinfo='text'
)])

# Update layout
fig.update_layout(
    title='3D Plot of DOM Positions',
    scene=dict(
        xaxis_title='X Position',
        yaxis_title='Y Position',
        zaxis_title='Z Position'
    ),
    width=750,
    height=600
)

fig.show()